#

<div align=center>
<img src="https://uol.unifor.br/acesso/app/autenticacao/assets/img/logos/icon-unifor.svg" width=45 height=45>

<br><br>
<font size=5 color='black'><strong>MBA Ciência de dados:</strong> Estatística descritiva

<strong>Projeto:</strong> Titanic

<strong>Autoria:</strong> Heitor Teixeira

</div>

## <font color=darkblue> 1. Imports e declaração de constantes

### <font color=steelblue> 1.1 Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, gaussian_kde
import numpy as np
import requests


### <font color=steelblue> 1.2 Constantes

Declarar constantes em uma unica celula facilita a manutenção de notebooks longos.<br>Alguns benefícios:
1. Melhorar a legibilidade do codigo
2. As constantes podem ser esquema de cores para padronizar sempre as mesmas cores para determinadas classes, paths de arquivos e datasets e etc.
3. posso modifica-las apenas aqui e servir para o codigo inteiro


In [ ]:
# paths dos arquivos originais professor + Bank of England
PATH_TITANIC = 'datasets/titanic.csv'
PATH_BOE = 'datasets/extras/a-millennium-of-macroeconomic-data-for-the-uk.xlsx'

# paths dos arquivos feitos na feature engineering
PATH_BOE_CPI = 'datasets/boe_cpi.csv'  
PATH_WB_CPI  = 'datasets/wb_cpi.csv'   

# endpoint de APIs
API_WORLD_BANK = "https://api.worldbank.org/v2/country/GBR/indicator/FP.CPI.TOTL"

# constantes para  variaveis qualitativas e cabecalho
LABELS_CLASSE = {1: 'Primeira Classe', 2: 'Segunda Classe', 3: 'Terceira Classe'}

LABELS_GENERO = {'male': 'Masculino', 'female': 'Feminino'}

LABELS_SOBREVIVENTE = {0: 'Não Sobreviveu', 1: 'Sobreviveu'}

MAP_COLUMNS = {
    'Survived': 'Sobreviveu',
    'Pclass': 'Classe',
    'Name': 'Nome',
    'Sex': 'Genero',
    'Age': 'Idade',
    'Siblings/Spouses Aboard': 'Irmãos/Cônjuges a Bordo',
    'Parents/Children Aboard': 'Pais/Filhos a Bordo',
    'Fare': 'Tarifa',
}

## <font color=darkblue> 2. Carregamento e Preparação do Dataset

### <font color=steelblue> 2.1 Carregando o dataset original

**Observações**<br>
Esta análise considera `exclusivamente os passageiros do Titanic. Algumas decisões metodológicas foram adotadas para garantir maior clareza e consistência nos resultados:

1. Registros com `Fare = 0` foram excluídos, pois não é possível distinguir se pertencem a passageiros ou tripulantes, e sua presença poderia distorcer métricas e visualizações.
2. As variáveis qualitativas já são carregadas com mapeamento pelos valores definidos nas constantes, padronizando rótulos em tabelas e gráficos sem necessidade de configuração adicional.
3. Por essas razões, pequenas variações nos cálculos de média, mediana, desvio padrão e outras medidas em relação a outras análises do mesmo dataset são esperadas.

In [3]:
# carregando o dataset já renomeado com as constantes definidas na seçao 1.2
titanic_df = pd.read_csv(PATH_TITANIC, header=0, names=MAP_COLUMNS.values())

titanic_df = titanic_df[titanic_df['Tarifa']>0]

titanic_df['Classe'] = titanic_df['Classe'].map(LABELS_CLASSE)
titanic_df['Genero'] = titanic_df['Genero'].map(LABELS_GENERO)
titanic_df['Sobreviveu'] = titanic_df['Sobreviveu'].map(LABELS_SOBREVIVENTE)
titanic_df.tail(1)


,Sobreviveu,Classe,Nome,Genero,Idade,Irmãos/Cônjuges a Bordo,Pais/Filhos a Bordo,Tarifa
886,Não Sobreviveu,Terceira Classe,Mr. Patrick Dooley,Masculino,32.0,0,0,7.75


### <font color=steelblue> 2.2 Feature Engineering

**Observações:***<br>
Em aula, o professor falou da importancia de considerar tanto valores absolutos quanto relativos. No caso do Titanic, como o acidente ocorreu em 1912, as tarifas originais estão completamente defasadas: £1 de 1912 não equivale a £1 hoje. <br><br>
Para preservar a noção de grandeza e permitir uma interpretação mais realista dos preços, adicionei uma coluna com a tarifa corrigida pela inflação. Os índices utilizados foram:

1. `Bank of England` - A Millennium of Macroeconomic Data for the UK (v3.1), aba A47, coluna H. Índice composto de preços do Reino Unido, normalizado para 100 em 2016. Cobre o período de 1912 a 2016.
2. `World Bank` - utilizado para estender a correção de 2016 até os anos mais recentes, complementando a série histórica do BoE.


#### <font color=slategray> 2.2.1 Aquisição de dados externos

In [ ]:
'''
essa celula é pra rodar só uma vez e depois comentar.
ela salva os datasets obtidos de APIs e dados externos para nao precisar puxar toda execucao


# extrai do Excel e salva em csv para evitar reler o arquivo de 27MB a cada execução
boe_df = pd.read_excel(
    PATH_BOE,
    sheet_name='A47. Wages and prices',
    header=6,       # as 6 primeiras linhas são metadados; a linha 6 é o cabeçalho real
    usecols=[0, 3], # col 0 = ano, col 3 = CPI preferred measure (GB/UK, base 2015=100)
)
boe_df.columns = ['Ano', 'Indice_CPI']
boe_df = boe_df.dropna(subset=['Ano', 'Indice_CPI'])
boe_df['Ano'] = boe_df['Ano'].astype(int)
boe_df.to_csv(PATH_BOE_CPI, index=False)

# obtém CPI de 2016 até 2024 via API e salva em csv pra nao chamar a API toda hora
# CPI é como se fosse a inflacao do UK -> Consumer Price Index
resp = requests.get(
    API_WORLD_BANK,
    params={"format": "json", "date": "2016:2024", "per_page": 20},
    timeout=10,
)
wb_df = pd.DataFrame(resp.json()[1])[["date", "value"]].dropna()
wb_df.columns = ['Ano', 'Indice_CPI']
wb_df['Ano'] = wb_df['Ano'].astype(int)
wb_df.to_csv(PATH_WB_CPI, index=False)

'''

#### <font color=slategray> 2.2.2 Fator de inflação atualizado

In [ ]:
# carrega os csvs salvos localmente, agora nao depende de API.
boe_df = pd.read_csv(PATH_BOE_CPI)
wb_df  = pd.read_csv(PATH_WB_CPI)

# fator BoE: corrige de 1912 até 2016
idx_1912  = boe_df.loc[boe_df['Ano'] == 1912, 'Indice_CPI'].values[0]
idx_2016  = boe_df.loc[boe_df['Ano'] == 2016, 'Indice_CPI'].values[0]
fator_boe = idx_2016 / idx_1912

# fator World Bank: estende a correção de 2016 até 2024
cpi_2016  = wb_df.loc[wb_df['Ano'] == 2016, 'Indice_CPI'].values[0]
cpi_2024  = wb_df.loc[wb_df['Ano'] == 2024, 'Indice_CPI'].values[0]
fator_wb  = cpi_2024 / cpi_2016

FATOR_INFLACAO = fator_boe * fator_wb
print(f"fator de inflação 1912 → 2024: {FATOR_INFLACAO:.2f}x")

fator de inflação 1912 → 2024: 107.14x
